<a href="https://colab.research.google.com/github/juliagermanosilva-pixel/Dados_poluicao_2025_2026/blob/main/Dados_poluicao_global_2025_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/ChiaviniK/dados_polui-o_2526.git

Cloning into 'dados_polui-o_2526'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), 297.43 KiB | 6.61 MiB/s, done.


In [4]:
import pandas as pd
import numpy as np
#método de extração - raw (cru - em português)
#ETL - Extract - Transform - Load
df_dados_raw = pd.read_csv('/content/dados_polui-o_2526/Global_Air_Pollution_Data_2025_2026.csv')
#ver os dados carregados
display(df_dados_raw)

,Date,City,Latitude,Longitude,PM2.5,PM10,NO2,SO2,CO,Ozone,Aerosol_Optical_Depth,AQI_Class
0,2025-11-08 00:00:00,"Lahore, Pakistan",31.5497,74.3436,81.0,82.7,94.0,16.4,1910.0,11.0,0.19,Unhealthy
1,2025-11-08 01:00:00,"Lahore, Pakistan",31.5497,74.3436,78.2,79.6,90.9,14.4,1413.0,10.0,0.19,Unhealthy
2,2025-11-08 02:00:00,"Lahore, Pakistan",31.5497,74.3436,75.4,76.8,85.0,12.8,1060.0,12.0,0.19,Unhealthy
3,2025-11-08 03:00:00,"Lahore, Pakistan",31.5497,74.3436,72.2,73.6,72.8,11.8,863.0,20.0,0.19,Unhealthy
4,2025-11-08 04:00:00,"Lahore, Pakistan",31.5497,74.3436,69.8,71.2,57.7,11.1,811.0,31.0,0.19,Unhealthy
...,...,...,...,...,...,...,...,...,...,...,...,...
17467,2026-02-06 19:00:00,"Jakarta, Indonesia",-6.2088,106.8456,49.4,49.9,48.6,39.2,3014.0,5.0,0.16,Unhealthy for Sensitive
17468,2026-02-06 20:00:00,"Jakarta, Indonesia",-6.2088,106.8456,59.6,60.5,47.3,41.6,2791.0,1.0,0.15,Unhealthy
17469,2026-02-06 21:00:00,"Jakarta, Indonesia",-6.2088,106.8456,61.5,62.2,41.5,42.5,2200.0,1.0,0.16,Unhealthy
17470,2026-02-06 22:00:00,"Jakarta, Indonesia",-6.2088,106.8456,61.5,62.2,36.0,42.0,1672.0,1.0,0.16,Unhealthy


In [5]:
#analise preliminar dos dados
#analise de estrutura de dados
df_dados_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17472 entries, 0 to 17471
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Date                   17472 non-null  object 
 1   City                   17472 non-null  object 
 2   Latitude               17472 non-null  float64
 3   Longitude              17472 non-null  float64
 4   PM2.5                  17472 non-null  float64
 5   PM10                   17472 non-null  float64
 6   NO2                    17472 non-null  float64
 7   SO2                    17472 non-null  float64
 8   CO                     17472 non-null  float64
 9   Ozone                  17472 non-null  float64
 10  Aerosol_Optical_Depth  17472 non-null  float64
 11  AQI_Class              17472 non-null  object 
dtypes: float64(9), object(3)
memory usage: 1.6+ MB


In [6]:
#resumo estatisticoo
df_dados_raw.describe().T

,count,mean,std,min,25%,50%,75%,max
Latitude,17472.0,23.292375,23.814690,-23.5505,16.305525,30.0818,40.106350,51.5074
Longitude,17472.0,43.056375,68.421659,-74.0060,-11.754175,75.7763,94.520775,116.4074
PM2.5,17472.0,66.060611,65.414505,0.3000,13.175000,45.5000,99.700000,430.3000
PM10,17472.0,69.263124,66.576392,0.4000,14.800000,48.9000,104.300000,434.0000
NO2,17472.0,38.458871,26.977070,0.0000,17.700000,31.8000,53.600000,185.1000
SO2,17472.0,30.292691,41.777128,0.0000,6.100000,14.4000,37.900000,439.3000
CO,17472.0,1264.659741,1484.401277,0.0000,278.000000,713.0000,1627.000000,13039.0000
Ozone,17472.0,54.746337,49.928268,0.0000,16.000000,45.0000,71.000000,288.0000
Aerosol_Optical_Depth,17472.0,0.325342,0.255573,0.0100,0.120000,0.2700,0.470000,2.9800


In [14]:
#vamos criar nossa engenharia de dados
def pipeline(df):
  #vamos criar nosso backup
  df_limpos = df.copy()
  #sempre que você trabalha com BI'sou SQL
  #padroziação dos nomes para uso (fazer tudo ficar em snack case)
  df_limpos.columns = [col.lower().replace(' ','_')
  for col in df_limpos.columns]
  #regra de negocio - traduzir para algo legivel
  condicoes = [
      (df_limpos['pm2.5'] <= 12),
      (df_limpos['pm2.5'] <= 35.5),
      (df_limpos['pm2.5'] <= 55.4),
      (df_limpos['pm2.5'] <= 150.4),
      (df_limpos['pm2.5'] > 150.5)
  ]
  escolha = ['Boa',
             'Moderado',
             'Insalubre para grupos sensiveis',
             'Insalubre',
             'Muito Insalubre/Perigoso'
  ]
  #criar um padrão para os dados
  df_limpos['timestamp'] = pd.to_datetime(df_limpos['date'])
  #agora vamos criar o campo para traduzir os numeros para as escolhas
  df_limpos['status_saude'] = np.select(condicoes, escolha, default='N/A')
  df_limpos['dia'] = df_limpos['timestamp'].dt.day
  df_limpos['Mês'] = df_limpos['timestamp'].dt.month
  df_limpos['Ano'] = df_limpos['timestamp'].dt.year
  df_limpos['Data'] = df_limpos['timestamp'].dt.date
  df_limpos['Hora'] = df_limpos['timestamp'].dt.hour
  return df_limpos

In [15]:
df_poluicao_final = pipeline(df_dados_raw)
df_poluicao_final.head(50)

,date,city,latitude,longitude,pm2.5,pm10,no2,so2,co,ozone,aerosol_optical_depth,aqi_class,timestamp,status_saude,dia,Mês,Ano,Data,Hora
0,2025-11-08 00:00:00,"Lahore, Pakistan",31.5497,74.3436,81.0,82.7,94.0,16.4,1910.0,11.0,0.19,Unhealthy,2025-11-08 00:00:00,Insalubre,8,11,2025,2025-11-08,0
1,2025-11-08 01:00:00,"Lahore, Pakistan",31.5497,74.3436,78.2,79.6,90.9,14.4,1413.0,10.0,0.19,Unhealthy,2025-11-08 01:00:00,Insalubre,8,11,2025,2025-11-08,1
2,2025-11-08 02:00:00,"Lahore, Pakistan",31.5497,74.3436,75.4,76.8,85.0,12.8,1060.0,12.0,0.19,Unhealthy,2025-11-08 02:00:00,Insalubre,8,11,2025,2025-11-08,2
3,2025-11-08 03:00:00,"Lahore, Pakistan",31.5497,74.3436,72.2,73.6,72.8,11.8,863.0,20.0,0.19,Unhealthy,2025-11-08 03:00:00,Insalubre,8,11,2025,2025-11-08,3
4,2025-11-08 04:00:00,"Lahore, Pakistan",31.5497,74.3436,69.8,71.2,57.7,11.1,811.0,31.0,0.19,Unhealthy,2025-11-08 04:00:00,Insalubre,8,11,2025,2025-11-08,4
5,2025-11-08 05:00:00,"Lahore, Pakistan",31.5497,74.3436,66.8,68.2,45.0,11.1,1088.0,43.0,0.20,Unhealthy,2025-11-08 05:00:00,Insalubre,8,11,2025,2025-11-08,5
6,2025-11-08 06:00:00,"Lahore, Pakistan",31.5497,74.3436,71.2,72.6,51.0,11.4,2069.0,36.0,0.20,Unhealthy,2025-11-08 06:00:00,Insalubre,8,11,2025,2025-11-08,6
7,2025-11-08 07:00:00,"Lahore, Pakistan",31.5497,74.3436,80.3,82.0,58.7,11.9,3380.0,28.0,0.21,Unhealthy,2025-11-08 07:00:00,Insalubre,8,11,2025,2025-11-08,7
8,2025-11-08 08:00:00,"Lahore, Pakistan",31.5497,74.3436,79.6,81.0,59.7,13.0,4085.0,34.0,0.21,Unhealthy,2025-11-08 08:00:00,Insalubre,8,11,2025,2025-11-08,8
9,2025-11-08 09:00:00,"Lahore, Pakistan",31.5497,74.3436,71.9,73.1,47.8,15.5,3616.0,68.0,0.20,Unhealthy,2025-11-08 09:00:00,Insalubre,8,11,2025,2025-11-08,9
